The Walkthrough Architecture

                 walkthrough.ipynb
                        │
─────────────────────────────────────────────────────
                        │
         Configure logging & choose model
                        │
                        ▼
               Load HuggingFace Model
                        │
                        ▼
              Wrap with jlens.from_hf()
                        │
                        ▼
            Load Pretrained Jacobian Lens
                        │
                        ▼
          Apply Jacobian Lens to a Prompt
                        │
                        ▼
      Compare Logit Lens vs Jacobian Lens
                        │
                        ▼
          Interactive Visualization
                        │
                        ▼
        Train Your Own Jacobian Lens

# Jacobian lens — walkthrough

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation. 

In [2]:
import jlens

jlens.configure_logging()

MODEL_NAME = "gpt2"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

# we will fit our own jacobian lens later.
# For now don't load any pretrained lens.

print("Model:", MODEL_NAME)

Model: gpt2


## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface

In [ ]:
import torch
import transformers

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
).to(device)

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

model = jlens.from_hf(hf_model, tokenizer)

print(model)

In [6]:
import jlens

jlens.configure_logging()

In [7]:
print(model)

HFLensModel(GPT2LMHeadModel, n_layers=12, d_model=768)


## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [8]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=50)
print(prompts)

[' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players take control of a military unit and take part in missions against enemy forces . Stories are told through comic book @-@ like panels with animated character portraits 

In [9]:
lens = jlens.fit(
    model,
    prompts,
    dim_batch=16,
    max_seq_len=128,
    checkpoint_path="ckpt.pt",
)

lens.save("jacobian_lens.pt")

[    36s + 36.19s] fit: n_layers=12 d_model=768, fitting 11 source layers (target=L11) on 50 prompts
[    36s +  0.13s]   resuming from checkpoint: 59/50 prompts processed
[    36s +  0.11s] fit: done, 59 prompts


In [10]:
import os 

print(os.path.exists("jacobian_lens.pt"))

True


In [11]:
# Load the lens we just trained
lens = jlens.JacobianLens.load("jacobian_lens.pt")

In [12]:
""".             JacobianLens
                  │
                  ├── n_prompts = 5
                  │
                  ├── d_model = 768
                  │
                  ├── source_layers
                  │     │
                  │     ├── 0
                  │     ├── 3
                  │     ├── 6
                  │     └── 9
                  │
                  └── jacobians
                        │
                        ├── J₀
                        │      768×768
                        │
                        ├── J₃
                        │      768×768
                        │
                        ├── J₆
                        │      768×768
                        │
                        └── J₉
                              768×768 """

'.             JacobianLens\n                  │\n                  ├── n_prompts = 5\n                  │\n                  ├── d_model = 768\n                  │\n                  ├── source_layers\n                  │     │\n                  │     ├── 0\n                  │     ├── 3\n                  │     ├── 6\n                  │     └── 9\n                  │\n                  └── jacobians\n                        │\n                        ├── J₀\n                        │      768×768\n                        │\n                        ├── J₃\n                        │      768×768\n                        │\n                        ├── J₆\n                        │      768×768\n                        │\n                        └── J₉\n                              768×768 '

In [13]:
print(lens)
print(model)
print(tokenizer)

JacobianLens(d_model=768, n_prompts=59, source_layers=[0..10] (11 layers))
HFLensModel(GPT2LMHeadModel, n_layers=12, d_model=768)
GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})


## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [14]:
prompt = "The largest planet in our solar system is"
layers = lens.source_layers

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-1])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-1], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")


L  0 logit-lens: [' not', ' also', ' now', ' still', ' a']
L  0 J-lens:     ['ModLoader', 'RIP', 'ahime', 'senal', ' ingred']
L  1 logit-lens: [' now', ' not', ' also', ' currently', ' still']
L  1 J-lens:     ['ModLoader', ' mathemat', 'senal', ' enthusi', ' ingred']
L  2 logit-lens: [' now', ' not', ' currently', ' still', ' also']
L  2 J-lens:     ['ModLoader', 'senal', ' mathemat', ' ingred', 'ccording']
L  3 logit-lens: [' now', ' currently', ' not', ' still', ' also']
L  3 J-lens:     [' confir', ' destro', 'ccording', ' tremend', ' alot']
L  4 logit-lens: [' now', ' not', ' currently', ' still', ' probably']
L  4 J-lens:     [' confir', ' tremend', 'ccording', ' ingred', ' princ']
L  5 logit-lens: [' now', ' probably', ' currently', ' still', ' undoubtedly']
L  5 J-lens:     [' tremend', ' confir', 'ccording', 'ModLoader', ' toget']
L  6 logit-lens: [' probably', ' now', ' indeed', ' still', ' definitely']
L  6 J-lens:     [' skelet', ' humankind', ' dwarf', ' tremend', ' dwar']

In [15]:
print(lens.source_layers)
print(lens.jacobians.keys())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10])


## 4. Render a slice page (inline)

`compute_slice` + `build_page` produce an interactive position × layer view of the lens's token ranks (the `?` in the corner explains the controls). `mode="embed"` inlines everything so the page is self-contained.

In [16]:
import gzip
import json

from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice, notebook_iframe

gloss = None

example = next(e for e in EXAMPLES if e.slug == "multihop")

prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    mask_display=False,      # <- change this
)

page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
)

notebook_iframe(page)

## 5. Render a slice page (served)

For longer prompts prefer `mode="fetch"`: `build_page` writes the data as sidecar files to `out_dir` and the page fetches rank files lazily on pin, so it stays small regardless of how many tokens are tracked.

In [17]:
import os
import threading

from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice

# GPT-2 doesn't need vocabulary glosses
gloss = None

# You can also try "multihop" later
example = next(e for e in EXAMPLES if e.slug == "multihop")

prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(
    model,
    lens,
    prompt,
    mask_display=False,      # <-- changed for GPT-2
)

out_dir = Path("slices") / example.slug

page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
    mode="fetch",
    out_dir=out_dir,
)

(out_dir / "index.html").write_text(page)

if "_jlens_httpd" not in globals():
    handler = partial(
        SimpleHTTPRequestHandler,
        directory=os.path.abspath("slices")
    )

    _jlens_httpd = HTTPServer(("127.0.0.1", 0), handler)

    threading.Thread(
        target=_jlens_httpd.serve_forever,
        daemon=True
    ).start()

print(
    f"Open in browser:\n"
    f"http://localhost:{_jlens_httpd.server_address[1]}/{example.slug}/"
)

Open in browser:
http://localhost:63400/multihop/


### More to explore

A few more prompts are bundled in `jlens.examples.EXAMPLES` — change the `slug` above and see what surfaces, or try a prompt of your own.

In [18]:
for e in EXAMPLES:
    print(f"{e.slug:>24}  {e.section}")
    

                multihop  Multi-hop reasoning
        modulation-topic  Voluntary modulation: topic
   modulation-arithmetic  Voluntary modulation: arithmetic
              ascii-face  ASCII face
              off-by-one  Bug in code
           overdose-flag  Overdose flag
           greatest-fear  Greatest fear (don't say it)
               blackmail  Agentic Misalignment (blackmail honeypot)


## 6. Fitting

`fit(model, prompts)` computes `J_l` over the supplied prompts. 100 prompts is enough for a usable lens; the released lenses use 1000. `dim_batch` is the memory knob — each prompt does `ceil(d_model / dim_batch)` backward passes on a retained graph.

In [ ]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=100)
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128, checkpoint_path="ckpt.pt"
)
lens.save("jacobian_lens.pt")

[    51s + 15.23s] fit: n_layers=12 d_model=768, fitting 11 source layers (target=L11) on 100 prompts
[    51s +  0.03s]   resuming from checkpoint: 59/100 prompts processed
127.0.0.1 - - [04/Aug/2026 11:59:35] "GET /multihop/ HTTP/1.1" 200 -
127.0.0.1 - - [04/Aug/2026 11:59:36] "GET /multihop/meta.json HTTP/1.1" 200 -
127.0.0.1 - - [04/Aug/2026 11:59:37] "GET /multihop/slice.bin HTTP/1.1" 200 -
127.0.0.1 - - [04/Aug/2026 11:59:37] code 404, message File not found
127.0.0.1 - - [04/Aug/2026 11:59:37] "GET /favicon.ico HTTP/1.1" 404 -
[  8m41s +469.77s]   prompt 60/100  seq_len=128 n_valid=111  470s  max||J||/sqrt(d)=1.709  max_d_mean=1.24e-02
[  9m56s + 74.89s]   prompt 61/100  seq_len=128 n_valid=111  75s  max||J||/sqrt(d)=1.720  max_d_mean=1.26e-02
[ 12m09s +133.21s]   prompt 62/100  seq_len=128 n_valid=111  132s  max||J||/sqrt(d)=1.863  max_d_mean=1.75e-02
[ 13m20s + 70.58s]   prompt 63/100  seq_len=128 n_valid=111  70s  max||J||/sqrt(d)=1.861  max_d_mean=1.47e-02
[ 15m39s +139.18s]